# Greenhouse Gas Reports: Fire CO₂ with Cortex Search

*An end-to-end workflow with AI_PARSE_DOCUMENT, Cortex Search, and AI_COMPLETE*

This notebook parses three WMO greenhouse gas bulletins in AGENTIC mode, indexes their page text, visual summaries, and chart details, and answers a question about Amazon fire CO₂ emissions in 2016 and 2024 from bulletin No. 21 with `claude-sonnet-5-5`.

The three PDFs come from `greenhouse_report.zip`. Upload them to the `GHG_REPORTS` stage before running the parse cells.

In [ ]:
%%sql -r ghg_setup
CREATE DATABASE IF NOT EXISTS VHOL_DOCUMENT_AI_DB;
CREATE SCHEMA IF NOT EXISTS VHOL_DOCUMENT_AI_DB.DOCUMENT_AI;

USE DATABASE VHOL_DOCUMENT_AI_DB;
USE SCHEMA DOCUMENT_AI;

SELECT CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE();

In [ ]:
%%sql -r ghg_stage
CREATE STAGE IF NOT EXISTS GHG_REPORTS
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
    DIRECTORY = (ENABLE = TRUE);

### Upload and Check the PDFs

1. Download and unzip [greenhouse_report.zip](https://github.com/Snowflake-Labs/sfquickstarts/blob/master/site/sfguides/src/document-intelligence-ai-functions/assets/greenhouse_report.zip). Upload the individual PDFs, not the ZIP archive.
2. In Snowsight, go to **Ingestion > Add Data > Load files into a Stage** and upload the three PDFs to `GHG_REPORTS`: `GHG-20_en.pdf`, `GHG-21_en.pdf`, and `GHG-22_en.pdf`.
3. Run the next cell to refresh the directory table and confirm all three files are listed.

In [ ]:
%%sql -r ghg_refresh
ALTER STAGE GHG_REPORTS REFRESH;

LIST @GHG_REPORTS;

## Register the Bulletins

The bulletin number is read from the `GHG-NN` part of each filename. The query should return three rows: bulletins 20, 21, and 22.

In [ ]:
%%sql -r ghg_files
CREATE OR REPLACE TEMP TABLE GHG_FILES AS
SELECT
    relative_path,
    SPLIT_PART(relative_path, '/', -1) AS filename,
    REGEXP_SUBSTR(relative_path, 'GHG-([0-9]+)', 1, 1, 'e', 1) AS bulletin,
    size AS file_size_bytes,
    last_modified
FROM DIRECTORY(@GHG_REPORTS)
WHERE relative_path ILIKE '%.pdf';

SELECT * FROM GHG_FILES ORDER BY bulletin;

## Parse the Bulletins with AGENTIC Mode
Duration: 10 minutes

`AI_PARSE_DOCUMENT` in AGENTIC mode returns page text and a visual interpretation for each image. `page_split` returns one record per page. The final `TRUE` returns error details in the result instead of failing the query.

This step calls the model for each bulletin and can take several minutes.

In [ ]:
%%sql -r ghg_parse
CREATE OR REPLACE TEMP TABLE GHG_PARSED AS
SELECT
    relative_path,
    filename,
    bulletin,
    AI_PARSE_DOCUMENT(
        TO_FILE('@GHG_REPORTS', relative_path),
        {'mode': 'AGENTIC', 'page_split': TRUE},
        TRUE
    ) AS raw_parse
FROM GHG_FILES;

SELECT
    filename,
    bulletin,
    ARRAY_SIZE(raw_parse:value:pages) AS returned_pages,
    raw_parse:error AS call_error,
    raw_parse:value:error AS document_error
FROM GHG_PARSED
ORDER BY bulletin;

## Split Pages, Visual Summaries, and Chart Details

The parsed output is split into one row per page and one row per image. Images that return an error are skipped. Each image's structured details are kept so the chart series can be indexed.

In [ ]:
%%sql -r ghg_pages
CREATE OR REPLACE TEMP TABLE GHG_PAGES AS
SELECT
    r.relative_path,
    r.filename,
    r.bulletin,
    p.value:index::NUMBER + 1 AS pdf_page_number,
    p.value:content::VARCHAR AS page_content,
    p.value:images AS images
FROM GHG_PARSED AS r,
     LATERAL FLATTEN(INPUT => r.raw_parse:value:pages) AS p
WHERE NULLIF(r.raw_parse:error::VARCHAR, 'null') IS NULL;

CREATE OR REPLACE TEMP TABLE GHG_VISUALS AS
SELECT
    p.relative_path,
    p.filename,
    p.bulletin,
    p.pdf_page_number,
    i.index AS image_index,
    i.value:results:image_type::VARCHAR AS image_type,
    i.value:results:summary::VARCHAR AS summary,
    i.value:results:details AS details
FROM GHG_PAGES AS p,
     LATERAL FLATTEN(INPUT => p.images) AS i
WHERE NULLIF(TRIM(i.value:results:summary::VARCHAR), '') IS NOT NULL
  AND COALESCE(LOWER(i.value:results:description::VARCHAR), '') <> 'error';

SELECT bulletin, COUNT(*) AS pages FROM GHG_PAGES GROUP BY bulletin ORDER BY bulletin;

SELECT bulletin, COUNT(*) AS visuals_with_summary FROM GHG_VISUALS GROUP BY bulletin ORDER BY bulletin;

## Check the Chart Details for Bulletin 21, Page 6

Page 6 of bulletin 21 has the Amazon CO₂ chart. The query shows each image on that page with its summary and chart details, and counts the chart details that mention both 2016 and 2024. Those values are read from the chart, so they are estimates. If the details do not include the years you need, the answer relies on the printed text instead.

In [ ]:
%%sql -r dataframe_1
SELECT
    pdf_page_number,
    image_index,
    image_type,
    summary,
    LEFT(TO_JSON(details), 1200) AS details_preview
FROM GHG_VISUALS
WHERE bulletin = '21'
  AND pdf_page_number = 6
ORDER BY image_index;

In [ ]:
%%sql -r ghg_figure11
SELECT
    COUNT(*) AS details_mention_2016_and_2024
FROM GHG_VISUALS
WHERE bulletin = '21'
  AND pdf_page_number = 6
  AND TO_JSON(details) LIKE '%2016%'
  AND TO_JSON(details) LIKE '%2024%';

## Build the Search Content

Index three kinds of content: the page text, the visual summaries, and the chart details. Each chunk repeats its bulletin number, PDF page, and content type, so a search can be restricted to one bulletin and traced back to its page.

In [ ]:
%%sql -r ghg_chunks
CREATE OR REPLACE TEMP TABLE GHG_SEARCH_UNITS AS
SELECT relative_path, filename, bulletin, pdf_page_number,
       'text'::VARCHAR AS content_type,
       page_content AS body
FROM GHG_PAGES
WHERE NULLIF(TRIM(page_content), '') IS NOT NULL
UNION ALL
SELECT relative_path, filename, bulletin, pdf_page_number,
       'visual_summary'::VARCHAR AS content_type,
       summary AS body
FROM GHG_VISUALS
UNION ALL
SELECT
    v.relative_path,
    v.filename,
    v.bulletin,
    v.pdf_page_number,
    'visual_details'::VARCHAR AS content_type,
    LISTAGG(d.path || ': ' || d.value::VARCHAR, '\n') WITHIN GROUP (ORDER BY d.path) AS body
FROM GHG_VISUALS AS v,
     LATERAL FLATTEN(INPUT => v.details, RECURSIVE => TRUE) AS d
WHERE TYPEOF(d.value) NOT IN ('OBJECT', 'ARRAY', 'NULL_VALUE')
  AND d.path NOT LIKE '%value_range%'
  AND d.path NOT LIKE '%value_precision%'
GROUP BY v.relative_path, v.filename, v.bulletin, v.pdf_page_number, v.image_index;

CREATE OR REPLACE TABLE GHG_SEARCH_CHUNKS
    DATA_RETENTION_TIME_IN_DAYS = 1 AS
SELECT
    MD5(u.relative_path || '|' || u.pdf_page_number::VARCHAR || '|' ||
        u.content_type || '|' || c.index::VARCHAR) AS chunk_id,
    u.relative_path,
    u.filename,
    u.bulletin,
    u.pdf_page_number,
    u.content_type,
    'Report: WMO Greenhouse Gas Bulletin No. ' || u.bulletin ||
    '. PDF page: ' || u.pdf_page_number::VARCHAR ||
    '. Content: ' || u.content_type || CHR(10) || c.value::VARCHAR AS search_text
FROM GHG_SEARCH_UNITS AS u,
     LATERAL FLATTEN(INPUT => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
         u.body, 'markdown', 1400, 150
     )) AS c
WHERE NULLIF(TRIM(c.value::VARCHAR), '') IS NOT NULL;

ALTER TABLE GHG_SEARCH_CHUNKS SET CHANGE_TRACKING = TRUE;

SELECT bulletin, content_type, COUNT(*) AS chunks
FROM GHG_SEARCH_CHUNKS
GROUP BY bulletin, content_type
ORDER BY bulletin, content_type;

## Create the Cortex Search Service

The service uses the warehouse selected in the notebook and filters on bulletin, content type, and filename. The index builds after creation, which can take a short time. If the retrieval below returns nothing right away, wait a minute and run the DESCRIBE statement again to check that `indexing_state` is `ACTIVE`.

In [ ]:
%%sql -r ghg_search_service
-- Replaces only the Search service created by this notebook.
DROP CORTEX SEARCH SERVICE IF EXISTS GHG_BULLETIN_SEARCH;

EXECUTE IMMEDIATE $$
DECLARE
    wh VARCHAR DEFAULT CURRENT_WAREHOUSE();
    ddl VARCHAR;
    missing_warehouse EXCEPTION (-20001, 'Select a lab warehouse before creating Search.');
BEGIN
    IF (wh IS NULL) THEN
        RAISE missing_warehouse;
    END IF;
    ddl := 'CREATE OR REPLACE CORTEX SEARCH SERVICE GHG_BULLETIN_SEARCH '
        || 'ON search_text ATTRIBUTES bulletin, content_type, filename '
        || 'WAREHOUSE = "' || REPLACE(wh, '"', '""') || '" '
        || 'TARGET_LAG = ''1 hour'' INITIALIZE = ON_CREATE '
        || 'AS SELECT chunk_id, search_text, relative_path, filename, '
        || 'bulletin, pdf_page_number, content_type FROM GHG_SEARCH_CHUNKS';
    EXECUTE IMMEDIATE :ddl;
    RETURN 'Created GHG_BULLETIN_SEARCH.';
END;
$$;

DESCRIBE CORTEX SEARCH SERVICE GHG_BULLETIN_SEARCH;

## Retrieve Passages for the Question

The question is searched within bulletin 21 only. Each passage is shown with the chart details from the same page, so the figure 11 series reach the answer step.

In [ ]:
%%sql -r ghg_hits
WITH hits AS (
    SELECT
        r.value:FILENAME::VARCHAR AS filename,
        r.value:BULLETIN::VARCHAR AS bulletin,
        r.value:PDF_PAGE_NUMBER::NUMBER AS pdf_page_number,
        r.value:CONTENT_TYPE::VARCHAR AS content_type,
        r.value:SEARCH_TEXT::VARCHAR AS passage
    FROM (
        SELECT PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'GHG_BULLETIN_SEARCH',
            '{"query": "Amazon fire CO2 emissions between 2016 and 2024", "columns": ["SEARCH_TEXT", "FILENAME", "BULLETIN", "PDF_PAGE_NUMBER", "CONTENT_TYPE"], "filter": {"@eq": {"BULLETIN": "21"}}, "limit": 6}'
        )) AS response
    ) AS s,
    LATERAL FLATTEN(INPUT => s.response:results) AS r
),
details AS (
    SELECT bulletin, pdf_page_number, LISTAGG(search_text, '\n') WITHIN GROUP (ORDER BY chunk_id) AS chart_details
    FROM GHG_SEARCH_CHUNKS
    WHERE content_type = 'visual_details'
    GROUP BY bulletin, pdf_page_number
)
SELECT h.filename, h.bulletin, h.pdf_page_number, h.content_type, h.passage, d.chart_details
FROM hits AS h
LEFT JOIN details AS d
  ON d.bulletin = h.bulletin
 AND d.pdf_page_number = h.pdf_page_number
ORDER BY h.pdf_page_number;

## Answer the Question with AI_COMPLETE

The answer step passes the retrieved passages and chart details to `claude-sonnet-5-5`. The model compares fire CO₂ emissions in 2016 and 2024, cites each claim by bulletin, PDF page, and source type (text, visual summary, or visual details), and says when a value comes from a chart description rather than printed text.

In [ ]:
%%sql -r ghg_answer
WITH context AS (
    SELECT
        LISTAGG(
            'Bulletin ' || bulletin || ', PDF page ' || pdf_page_number::VARCHAR || ' (' || content_type || '):\n' || passage ||
            IFF(chart_details IS NULL, '', '\nChart details for page ' || pdf_page_number::VARCHAR || ':\n' || chart_details),
            '\n---\n'
        ) WITHIN GROUP (ORDER BY pdf_page_number) AS passages
    FROM {{ghg_hits}}
)
SELECT AI_COMPLETE(
    'claude-sonnet-5-5',
    'You answer questions about WMO greenhouse gas bulletins using only the passages below. Question: Compare Amazon fire CO2 emissions between 2016 and 2024. Which year is higher, and by roughly how much? Give approximate values and say they are read from charts when they are. Cite each claim as [Bulletin N, PDF page P, source], where source is the content label of the passage you used: text, visual_summary, or visual_details. Say when a value comes from a chart description rather than printed text. If the passages do not contain the answer, say what is missing.\n\nPassages:\n' || passages
) AS answer
FROM context;

## Cleanup

Both lab notebooks create their objects in `VHOL_DOCUMENT_AI_DB`. Dropping the database removes everything this notebook created: the stage and uploaded PDFs, the Cortex Search service, and the tables.

Do not run this against a shared event database. In a shared account, ask the host which objects you can remove.

In [ ]:
%%sql -r ghg_cleanup
DROP DATABASE IF EXISTS VHOL_DOCUMENT_AI_DB;

## Conclusion

You parsed three WMO greenhouse gas bulletins with AGENTIC mode, indexed their page text, visual summaries, and chart details, and asked a question that needed both printed text and chart values. The answer gives approximate values and cites each claim by bulletin, PDF page, and source type.